In [ ]:
import os
import shutil
import random
from pathlib import Path
import numpy as np
import tensorflow as tf
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, precision_recall_curve
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

device = "/GPU:0" if tf.config.list_physical_devices('GPU') else "/CPU:0"
print(f"Using device: {device}")
print(f"TensorFlow version: {tf.__version__}")

In [ ]:
RAW_DATA_DIR = Path('metal_nut/metal_nut')
PROCESSED_DATA_DIR = Path('data_processed')

def reorganize_dataset(split_ratio=0.8):
    if PROCESSED_DATA_DIR.exists():
        print(f"Directory {PROCESSED_DATA_DIR} already exists.")
        return    

    for split in ['train', 'val']:
        for class_name in ['good', 'defect']:
            (PROCESSED_DATA_DIR / split / class_name).mkdir(parents=True, exist_ok=True)

    raw_train_good = list((RAW_DATA_DIR / 'train' / 'good').glob('*.png'))
    raw_test_good = list((RAW_DATA_DIR / 'test' / 'good').glob('*.png'))
    all_good = raw_train_good + raw_test_good
    
    random.shuffle(all_good)
    split_idx = int(len(all_good) * split_ratio)
    train_good = all_good[:split_idx]
    val_good = all_good[split_idx:]

    for img_path in train_good:
        new_name = f"{img_path.parent.parent.name}_{img_path.parent.name}_{img_path.name}"
        shutil.copy(img_path, PROCESSED_DATA_DIR / 'train' / 'good' / new_name)
    for img_path in val_good:
        new_name = f"{img_path.parent.parent.name}_{img_path.parent.name}_{img_path.name}"
        shutil.copy(img_path, PROCESSED_DATA_DIR / 'val' / 'good' / new_name)

    all_defect = []
    test_dir = RAW_DATA_DIR / 'test'
    if test_dir.exists():
        for defect_type_dir in test_dir.iterdir():
            if defect_type_dir.is_dir() and defect_type_dir.name != 'good':
                all_defect.extend(list(defect_type_dir.glob('*.png')))
                
    random.shuffle(all_defect)
    split_idx = int(len(all_defect) * split_ratio)
    train_defect = all_defect[:split_idx]
    val_defect = all_defect[split_idx:]

    for img_path in train_defect:
        new_name = f"{img_path.parent.name}_{img_path.name}"
        shutil.copy(img_path, PROCESSED_DATA_DIR / 'train' / 'defect' / new_name)
    for img_path in val_defect:
        new_name = f"{img_path.parent.name}_{img_path.name}"
        shutil.copy(img_path, PROCESSED_DATA_DIR / 'val' / 'defect' / new_name)

    print(f"Train Good: {len(train_good)}, Train Defect: {len(train_defect)}")
    print(f"Val Good: {len(val_good)}, Val Defect: {len(val_defect)}")

reorganize_dataset()

In [ ]:
img_size = 224
batch_size = 32

train_dataset = tf.keras.utils.image_dataset_from_directory(
    PROCESSED_DATA_DIR / 'train',
    image_size=(img_size, img_size),
    batch_size=batch_size,
    label_mode='binary',
    shuffle=True
)

val_dataset = tf.keras.utils.image_dataset_from_directory(
    PROCESSED_DATA_DIR / 'val',
    image_size=(img_size, img_size),
    batch_size=batch_size,
    label_mode='binary',
    shuffle=False
)

class_names = train_dataset.class_names

num_defect = len(list((PROCESSED_DATA_DIR / 'train' / 'defect').glob('*.png')))
num_good = len(list((PROCESSED_DATA_DIR / 'train' / 'good').glob('*.png')))

train_dataset = train_dataset.map(lambda x, y: (x, 1.0 - y))
val_dataset = val_dataset.map(lambda x, y: (x, 1.0 - y))

train_dataset = train_dataset.prefetch(buffer_size=tf.data.AUTOTUNE)
val_dataset = val_dataset.prefetch(buffer_size=tf.data.AUTOTUNE)

print(f"Imbalance Distribution: {num_good} Good, {num_defect} Defect")

In [ ]:
def get_augmentation_layer():
    return tf.keras.Sequential([
        tf.keras.layers.RandomFlip("horizontal_and_vertical"),
        tf.keras.layers.RandomRotation(0.04),
        tf.keras.layers.RandomContrast(0.2)
    ], name="data_augmentation")

def BaselineCNN():
    inputs = tf.keras.Input(shape=(224, 224, 3))
    x = get_augmentation_layer()(inputs)

    x = tf.keras.layers.Rescaling(1./255)(x)
    
    x = tf.keras.layers.Conv2D(16, (3, 3), activation='relu', padding='same')(x)
    x = tf.keras.layers.MaxPooling2D((2, 2))(x)
    
    x = tf.keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same')(x)
    x = tf.keras.layers.MaxPooling2D((2, 2))(x)
    
    x = tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = tf.keras.layers.MaxPooling2D((2, 2))(x)
    
    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(128, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)
    
    return tf.keras.Model(inputs, outputs, name="baseline_cnn")

def TransferLearningCNN(freeze_backbone=True):
    inputs = tf.keras.Input(shape=(224, 224, 3))
    x = get_augmentation_layer()(inputs)
    

    base_model = tf.keras.applications.EfficientNetB0(
        input_shape=(224, 224, 3),
        include_top=False,
        weights='imagenet'
    )
    if freeze_backbone:
        base_model.trainable = False
        
    x = base_model(x, training=not freeze_backbone)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(256, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)
    
    return tf.keras.Model(inputs, outputs, name="efficientnet_cnn")

In [ ]:
class_weight = {0: 1.0, 1: float(num_good) / max(num_defect, 1)}
print(f"Keras class weights: {class_weight}")

os.makedirs('saved_models', exist_ok=True)

baseline_model = BaselineCNN()
baseline_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=['accuracy']
)

checkpoint_baseline = tf.keras.callbacks.ModelCheckpoint(
    filepath='saved_models/baseline_best.keras',
    monitor='val_loss',
    save_best_only=True,
    mode='min',
    verbose=1
)

baseline_model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,
    class_weight=class_weight,
    callbacks=[checkpoint_baseline]
)

efficientnet_model = TransferLearningCNN(freeze_backbone=True)
efficientnet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=['accuracy']
)

checkpoint_efficientnet = tf.keras.callbacks.ModelCheckpoint(
    filepath='saved_models/efficientnet_best.keras',
    monitor='val_loss',
    save_best_only=True,
    mode='min',
    verbose=1
)

efficientnet_model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,
    class_weight=class_weight,
    callbacks=[checkpoint_efficientnet]
)

In [ ]:
def evaluate_model(model, name, threshold=0.5):
    all_preds = []
    all_probs = []
    all_labels = []
    
    for inputs, labels in val_dataset:
        probs = model.predict(inputs, verbose=0)
        preds = (probs >= threshold).astype(int)
        
        all_probs.extend(probs.flatten())
        all_preds.extend(preds.flatten())
        all_labels.extend(labels.numpy().flatten())
        
    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)
    
    precision = precision_score(all_labels, all_preds)
    recall = recall_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    
    print(f"\n{name} Metrics for the Defect Class")
    print(f"Precision: {precision:.4%}")
    print(f"Recall:    {recall:.4%}")
    print(f"F1 Score:  {f1:.4f}")
    

    cm = confusion_matrix(all_labels, all_preds)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Good', 'Defect'], yticklabels=['Good', 'Defect'], ax=ax1)
    ax1.set_ylabel('Actual Label')
    ax1.set_xlabel('Predicted Label')
    ax1.set_title(f'Confusion Matrix: {name}')
    

    precisions, recalls, _ = precision_recall_curve(all_labels, all_probs)
    ax2.plot(recalls, precisions, marker='.', label=name, color='darkorange')
    ax2.set_xlabel('Recall')
    ax2.set_ylabel('Precision')
    ax2.set_title(f'Precision-Recall Curve: {name}')
    ax2.legend()
    ax2.grid(True)
    
    plt.tight_layout()
    plt.show()

best_baseline = tf.keras.models.load_model('saved_models/baseline_best.keras')
best_efficientnet = tf.keras.models.load_model('saved_models/efficientnet_best.keras')

evaluate_model(best_baseline, "Scratch Baseline CNN")
evaluate_model(best_efficientnet, "EfficientNetB0 Transfer Learning")